# Probabilistic Circuit Structural Learning Demo

This notebook demonstrates the structural learning functionality
implemented for RX probabilistic circuits.

The demo follows the pruning and growing strategy described in:

Dang, M., Liu, A., and Van den Broeck, G.
"Sparse Probabilistic Circuits via Pruning and Growing."
NeurIPS 2022.

The process consists of:

1. Creating an initial probabilistic circuit.
2. Computing edge flows.
3. Removing low-contribution connections.
4. Growing the circuit by duplicating SumUnits.
5. Applying the complete structural adaptation process.

In [3]:
import sys
from pathlib import Path

ROOT = Path("../../").resolve()

sys.path.append(
    str(ROOT / "probabilistic_model" / "src")
)

sys.path.append(
    str(ROOT / "krrood" / "src")
)

sys.path.append(
    str(ROOT / "semantic_digital_twin" / "src")
)

In [4]:
import numpy as np

from probabilistic_model.probabilistic_circuit.rx.learning import (
    calculate_edge_flows,
    prune_edges,
    grow_nodes,
    sparse_pc_learning,
)

from probabilistic_model.probabilistic_circuit.rx.probabilistic_circuit import (
    ProbabilisticCircuit,
    SumUnit,
    ProductUnit,
    leaf,
    Continuous,
)

from probabilistic_model.distributions.gaussian import GaussianDistribution

## Creating a small probabilistic circuit

We create a simple Gaussian mixture model represented as a
probabilistic circuit.

The initial circuit contains two Gaussian components combined
by a SumUnit.

In [5]:
def create_demo_circuit():

    x = Continuous("x")

    pc = ProbabilisticCircuit()

    root = SumUnit(
        probabilistic_circuit=pc
    )

    p1 = ProductUnit(
        probabilistic_circuit=pc
    )

    p2 = ProductUnit(
        probabilistic_circuit=pc
    )

    l1 = leaf(
        GaussianDistribution(
            variable=x,
            location=0,
            scale=1,
        ),
        pc,
    )

    l2 = leaf(
        GaussianDistribution(
            variable=x,
            location=5,
            scale=1,
        ),
        pc,
    )

    p1.add_subcircuit(l1)

    p2.add_subcircuit(l2)

    root.add_subcircuit(
        p1,
        np.log(0.5)
    )

    root.add_subcircuit(
        p2,
        np.log(0.5)
    )

    return pc


circuit = create_demo_circuit()

circuit

ProbabilisticCircuit with 5 nodes and 4 edges

## 1. Computing Edge Flows

Edge flows estimate how much every weighted connection contributes
to the probability distribution.

Connections with low contribution are candidates for pruning.

In [6]:
data = np.array(
    [
        [0.0],
        [5.0],
    ]
)


flows = calculate_edge_flows(
    circuit,
    data,
)


flows

{(⊕, ⊗): 0.5, (⊕, ⊗): 0.5}

## 2. Pruning Low-Contribution Edges

Pruning removes unnecessary connections while keeping the most
important probability paths.

The remaining SumUnit weights are normalized afterwards.

In [7]:
before_edges = len(
    list(circuit.edges())
)


prune_edges(
    circuit,
    data,
    prune_fraction=0.2,
)


after_edges = len(
    list(circuit.edges())
)


print("Edges before pruning:", before_edges)
print("Edges after pruning:", after_edges)

Edges before pruning: 4
Edges after pruning: 4


## 3. Growing the Circuit

Growing increases the capacity of the probabilistic circuit.

Selected SumUnits are duplicated and their weights are slightly
perturbed to create new structural alternatives.

In [8]:
before_nodes = len(
    list(circuit.graph.nodes())
)


grow_nodes(
    circuit,
    fraction=0.5,
    noise_scale=1e-3,
)


after_nodes = len(
    list(circuit.graph.nodes())
)


print("Nodes before growing:", before_nodes)
print("Nodes after growing:", after_nodes)

Nodes before growing: 5
Nodes after growing: 7


## 4. Complete Structural Learning Pipeline

The complete algorithm alternates pruning and growing operations.

It automatically reduces unnecessary structure and increases
capacity where required by the data.

In [9]:
circuit = create_demo_circuit()


result = sparse_pc_learning(
    circuit,
    data,
    prune_fraction=0.2,
    grow_fraction=0.5,
    iterations=2,
)


result

ProbabilisticCircuit with 8 nodes and 11 edges

## When should structural learning be used?

Structural learning is useful when:

- the initial probabilistic circuit contains redundant structure,
- manually designing the architecture is difficult,
- the model needs to adapt its complexity to the data,
- a compact but expressive probabilistic circuit is desired.

Pruning improves efficiency by removing unnecessary connections,
while growing increases modelling capacity by introducing new
structural components.